# Netwoek anomaly detection project 
عرفان کریملو -
دانشگاه صدرا


## 1. creating virtual environment and installing dependencies 
first we have to make VENV : 
```
python -m venv <environment name>
```
---

then activating it : 
```
<environment name>/scripts/activate
```
--- 

and in the end easily download all dependecies :
```
pip install -r requirements.txt
```
---

## 2. importing libraries : 

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

sns.set_theme(style="whitegrid")


## 3. defining the path of project

In [2]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "synthetic_network_flows.csv"
RESULTS_PATH = PROJECT_ROOT / "results"

print(DATA_PATH)


c:\Users\Erfan\Desktop\NAD-project\data\raw\synthetic_network_flows.csv


## 4. Reading Dataset


In [3]:
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())
display(df.info())


Dataset shape: (20000, 18)


,Record_ID,Flow_Duration_ms,Total_Packets,Total_Bytes,Packets_per_Second,Bytes_per_Second,Avg_Packet_Size,Src_Port,Dst_Port,Active_Time_ms,Idle_Time_ms,TCP_Flag_Count,Unique_Dst_Ports,Failed_Connections,Packet_Size_Std,Connection_Rate,Label,Anomaly_Type
0,1,810,227,185465,280.126,228869.862,817.025,45201,80,622,188,4,5,0,110.598,1.700,Normal,Normal
1,2,1121,274,221237,244.341,197288.750,807.433,37268,53,936,185,3,3,1,167.095,2.748,Normal,Normal
2,3,351,60,36182,170.718,102948.116,603.029,18641,443,340,12,6,5,0,131.484,1.654,Normal,Normal
3,4,2793,282,241434,100.984,86457.727,856.149,48816,22,2505,288,2,3,0,140.818,1.432,Normal,Normal
4,5,1026,180,165035,175.450,160864.091,916.863,23397,443,846,180,2,1,0,137.469,1.364,Normal,Normal


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Record_ID           20000 non-null  int64  
 1   Flow_Duration_ms    20000 non-null  int64  
 2   Total_Packets       20000 non-null  int64  
 3   Total_Bytes         20000 non-null  int64  
 4   Packets_per_Second  20000 non-null  float64
 5   Bytes_per_Second    20000 non-null  float64
 6   Avg_Packet_Size     20000 non-null  float64
 7   Src_Port            20000 non-null  int64  
 8   Dst_Port            20000 non-null  int64  
 9   Active_Time_ms      20000 non-null  int64  
 10  Idle_Time_ms        20000 non-null  int64  
 11  TCP_Flag_Count      20000 non-null  int64  
 12  Unique_Dst_Ports    20000 non-null  int64  
 13  Failed_Connections  20000 non-null  int64  
 14  Packet_Size_Std     20000 non-null  float64
 15  Connection_Rate     20000 non-null  float64
 16  Labe

None

## 5. prepareing and checking columns 

In [ ]:
print(df.columns.tolist())
display(df["Label"].value_counts())


## 6. Data PreProcessing 

In [ ]:
metadata_columns = ["Record_ID", "Label", "Anomaly_Type"]

feature_columns = [
    column for column in df.columns
    if column not in metadata_columns
]

X = df[feature_columns].copy()
y = df["Label"].copy()

print("Number of features:", len(feature_columns))
print(feature_columns)


- if values where : "normal " and "anomaly" we do this : 

In [ ]:
if y.dtype == "object":
    y = y.map({
        "Normal": 0,
        "Anomaly": 1
    })

y = y.astype(int)


7. Training Model

In [ ]:
model = IsolationForest(
    n_estimators=200,
    contamination=0.10,
    random_state=42,
    n_jobs=-1
)

model.fit(X)


## 8. making predictaions

In [ ]:
raw_predictions = model.predict(X)
anomaly_scores = -model.decision_function(X)

predicted_labels = np.where(raw_predictions == -1, 1, 0)

predictions = pd.DataFrame({
    "Record_ID": df["Record_ID"],
    "Label": y,
    "Anomaly_Type": df["Anomaly_Type"],
    "Predicted_Label": predicted_labels,
    "Anomaly_Score": anomaly_scores
})

display(predictions.head())


## 9. Evaluation

In [ ]:
accuracy = accuracy_score(y, predicted_labels)
precision = precision_score(y, predicted_labels, zero_division=0)
recall = recall_score(y, predicted_labels, zero_division=0)
f1 = f1_score(y, predicted_labels, zero_division=0)

metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-Score"],
    "Value": [accuracy, precision, recall, f1]
})

display(metrics)


## 10. making the confuscation matrix

In [ ]:
cm = confusion_matrix(y, predicted_labels)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Normal", "Anomaly"],
    yticklabels=["Normal", "Anomaly"]
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()


## 11. Label Distribution 

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Label")
plt.title("Label Distribution")
plt.xlabel("Label")
plt.ylabel("Count")
plt.show()


## 12 . Anomaly Score Distribution

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(data=predictions, x="Anomaly_Score", hue="Label", bins=40)
plt.title("Anomaly Score Distribution")
plt.show()


## 13. anomally type anaalysis

In [ ]:
anomaly_type_analysis = (
    predictions[predictions["Label"] == 1]
    .groupby("Anomaly_Type")
    .agg(
        Total=("Label", "count"),
        Detected=("Predicted_Label", "sum")
    )
    .reset_index()
)

anomaly_type_analysis["Recall"] = (
    anomaly_type_analysis["Detected"] /
    anomaly_type_analysis["Total"]
)

display(anomaly_type_analysis)


## 14. Full Report

In [ ]:
print(classification_report(
    y,
    predicted_labels,
    target_names=["Normal", "Anomaly"],
    zero_division=0
))


## 15. Final Output and Save

In [ ]:
(RESULTS_PATH / "predictions").mkdir(parents=True, exist_ok=True)
(RESULTS_PATH / "tables").mkdir(parents=True, exist_ok=True)
(RESULTS_PATH / "figures").mkdir(parents=True, exist_ok=True)

predictions.to_csv(
    RESULTS_PATH / "predictions" / "traceable_predictions.csv",
    index=False
)

metrics.to_csv(
    RESULTS_PATH / "tables" / "notebook_evaluation_metrics.csv",
    index=False
)

anomaly_type_analysis.to_csv(
    RESULTS_PATH / "tables" / "notebook_anomaly_type_analysis.csv",
    index=False
)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Normal", "Anomaly"],
    yticklabels=["Normal", "Anomaly"]
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.savefig(
    RESULTS_PATH / "figures" / "notebook_confusion_matrix.png",
    dpi=150,
    bbox_inches="tight"
)
plt.show()

print("All outputs were saved successfully.")
